# Customer Segmentation using K-Means — EDA & ModelingThis notebook walks through the full pipeline end-to-end:1. Load transactions from SQLite2. Clean data3. Engineer RFM (Recency, Frequency, Monetary) features4. Explore distributions5. Find optimal k (elbow + silhouette)6. Fit K-Means and label business segments7. Visualize clusters and derive marketing recommendations

In [ ]:
import sys, ossys.path.append(os.path.join(os.getcwd(), '..', 'src'))import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom rfm import build_rfm_tablefrom clustering import prepare_features, evaluate_k_range, fit_kmeans, label_segments, run_pipeline, SEGMENT_RECOMMENDATIONSpd.set_option('display.max_columns', None)

## 1–3. Load, clean, and build RFM table

In [ ]:
rfm, clean_txns = build_rfm_table()print(f'Customers: {len(rfm)} | Clean transactions: {len(clean_txns)}')rfm.head()

## 4. Explore RFM distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))for ax, col in zip(axes, ['Recency', 'Frequency', 'Monetary']):    ax.hist(rfm[col], bins=30, color='#3b82f6', edgecolor='white')    ax.set_title(f'{col} distribution')plt.tight_layout()plt.show()

In [ ]:
rfm[['Recency', 'Frequency', 'Monetary']].describe()

## 5. Choose k — Elbow method & Silhouette scoreFeatures are log-transformed (Frequency, Monetary) to reduce skew, then standardized before clustering.

In [ ]:
scaled, scaler = prepare_features(rfm)eval_df = evaluate_k_range(scaled, k_range=range(2, 9))eval_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))axes[0].plot(eval_df['k'], eval_df['inertia'], marker='o')axes[0].set_title('Elbow Method'); axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')axes[1].plot(eval_df['k'], eval_df['silhouette'], marker='o', color='green')axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('k'); axes[1].set_ylabel('Score')plt.tight_layout()plt.show()

## 6. Fit K-Means (k=4) and label business segmentsk=4 is chosen to align directly with the business framing: VIP / Loyal / Potential / At-risk.

In [ ]:
result = run_pipeline(rfm, k=4)labeled_df = result['rfm_labeled']cluster_stats = result['cluster_stats']cluster_stats

In [ ]:
labeled_df['Segment'].value_counts()

## 7. Visualize clusters

In [ ]:
colors = {'VIP': '#22c55e', 'Loyal': '#3b82f6', 'Potential': '#eab308', 'At-risk': '#ef4444'}fig, ax = plt.subplots(figsize=(8, 6))for seg, group in labeled_df.groupby('Segment'):    ax.scatter(group['Frequency'], group['Monetary'], label=seg, alpha=0.6, c=colors.get(seg))ax.set_xlabel('Frequency (# purchases)')ax.set_ylabel('Monetary ($ total spend)')ax.set_title('Customer Segments: Frequency vs Monetary')ax.legend()plt.show()

## Business recommendations per segment

In [ ]:
for seg, info in SEGMENT_RECOMMENDATIONS.items():    if seg in labeled_df['Segment'].unique():        n = (labeled_df['Segment'] == seg).sum()        print(f"{info['emoji']} {seg} ({n} customers)")        print(f"  {info['description']}")        print(f"  Strategy: {info['strategy']}")        print()

## Export final labeled dataset

In [ ]:
labeled_df.to_csv('../data/customer_segments.csv', index=False)print('Saved to ../data/customer_segments.csv')